In [2]:
import pandas as pd
import requests
import os
from core.utils.calculadora_inflacao import CalculadoraFatorInflacao

# APOGESP
## Cálculo de impacto orçamentário das propostas reajuste salarial para os APPGGS

Esse notebook realiza a estimativa de impacto orçamentário das propostas de reajuste salarial feitas pela APOGESP para a campanha salarial de 2026 na Prefeitura de São Paulo.

São consideradas duas propostas principais:
1. Reajuste inflacionário, que se subdivide em:
    1. Reajuste da tabela original da carreira considerando a inflação acumulada desde as primeiras nomeações (jun/2026)
    2. Reajuste da tabela da carreira considerando a inflação acumulada na gestão Ricardo Nunes e a tabela no momento de início da gestão (jun/2021)
2. Equiparação com a tabela dos AMCI (ou seja, unificação da estrutura remuneratória do quadro QPPGG)


Em ambos os casos, a estimativa segue a seguinte metodologia:

1. Identificação dos APPGGs em exercício (incluso cedidos) segundo os últimos dados disponíveis no portal de dados abertos (dezembro de 2025),
2. Atualização dos dados do item 1 considerando o acréscimo dos 30 APPGGs recém-nomeados no nível 1, criando dados sintéticos (novas linhas na tabela)
3. Cálculo do salário base atual de cada APPGG considerando a tabela atual e o nível em que ele se encontra,
4. Cálculo dos encargos e benefícios que tem como base o salário atual como contribuição previdenciária obrigatória, auxílio transporte, auxílio alimentação etc.
5. Cálculo do salário proposto considerando o nível em que o APPGG se encontra atualmente e a tabela atualizada proposta
6. Cálculo dos encargos citados no item 6,
7. Somatória dos salários e encargos para cada APPGG considerando a tabela atual e a tabela proposta,
8. Cálculo da diferença entre a situação atual e a situação proposta conforme item 7,
9. Somatória da diferença por nível da carreira APPGG (p. ex., para o nível 1 a proposta implica em X mil reais mensais para a Prefeitura para o nível 2, y mil reais etc.)
10. Simulação de progressão da carreira considerando 18 meses para a "subida" de nível e projeção do impacto orçamentário anual até o final da gestão (2028)

O diagrama na imagem abaixo representa as etapas da metodologia.

![Grafo da metodologia](mermaid_graph.svg)

## Dados servidores

Nessa seção pegamos os dados dos servidores ativos mais atualizados (dezembro de 2025) do portal de dados abertos. Em seguida, identificamos os APPGGs e adicionamos dados sintéticos referentes aos APPGGs recém-nomeados.

In [3]:
URL_SERVIDORES = 'https://dados.prefeitura.sp.gov.br/dataset/bf5df0f4-4fb0-4a5e-b013-07d098cc7b1c/resource/e4c65839-3bc8-4035-b0a7-108ec8740536/download/verificado_ativos_05-01-2026_dez-2025in.csv'
file_dados = 'servidores_ativos.csv'

In [4]:
def download_dados_servidores(fname:str, url:str=URL_SERVIDORES)->str:

    if not os.path.exists(fname):
        print('Downloading data')
        with requests.get(url) as r:
            r.raise_for_status()
            content = r.content
            with open(fname, 'wb') as f:
                f.write(content)
            
            assert os.path.exists(fname)
            return fname
    return fname

In [5]:
file_dados = download_dados_servidores(file_dados)

In [6]:
df = pd.read_csv(file_dados, encoding='latin1', sep=';')

In [7]:
df.dtypes

REGISTRO                int64
VINCULO                 int64
NOME                      str
CARGO_BASICO              str
REF_CARGO_BAS             str
SEGMENTO                  str
GRUPO                     str
SUBGRUPO                  str
ESCOL_CARGO_BASICO        str
CARGO_COMISSAO            str
REF_CARGO_COM             str
ESCOL_CARGO_COMISSAO      str
DESIGNACAO                str
REF_DESIGNACAO            str
ESCOL_DESIGNACAO          str
JORNADA                   str
DATA_INICIO_EXERC         str
REL_JUR_ADM               str
SECRET_SUBPREF            str
SIGLA                     str
SETOR                     str
DISTRITO                  str
SUBPREFEITURA             str
ORGAO_EXT                 str
SEXO                      str
ANO_NASCIMENTO          int64
RACA_COR                  str
PCD                       str
dtype: object

In [8]:
df.describe()

,REGISTRO,VINCULO,ANO_NASCIMENTO
count,1.297360e+05,129736.000000,129736.000000
mean,7.926761e+06,1.506506,1976.357734
std,1.000625e+06,0.944192,10.229901
min,1.145541e+06,1.000000,1938.000000
25%,7.286054e+06,1.000000,1969.000000
50%,8.010154e+06,1.000000,1977.000000
75%,8.506738e+06,2.000000,1983.000000
max,9.536361e+06,17.000000,2007.000000


In [9]:
df.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,1145541,16,CLEUZA BORGES PEREIRA SILVA,ASSESSOR IV,CDA-4,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1949,PARDA,NAO
1,1160206,5,ADILSON DA SILVA,COORDENADOR II,CDA-6,NaN,QC,CARGO EM COMISSAO,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTO AMARO,SUB-SA,COORDENADORIA DE PLANEJAMENTO E DESENVOLVIMENT...,SANTO AMARO,SANTO AMARO,NaN,MASCULINO,1949,BRANCA,NAO
2,1160478,2,JULIO DE CARVALHO,FISCAL DE POSTURAS MUNICIPAIS NIVEL IV,QFPM15,NaN,QFPM,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTANA/TUCURUVI,SUB-ST,SUPERVISÃO TECNICA DE FISCALIZACAO,TUCURUVI,SANTANA/TUCURUVI,NaN,MASCULINO,1951,BRANCA,NAO
3,1161181,9,NEUSA PEDRAO NASSIR,ASSESSOR V,CDA-5,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA DO GOVERNO MUNICIPAL,SGM,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1947,BRANCA,NAO
4,1168185,4,MARIA HELENA DI VERNIERI CUPPARI,ASSISTENTE TECNICO EDUCACIONAL,QPE17A,NaN,QPE L. 14660/07 RGPS,CARGO EM COMISSAO,LICENCIATURA PLENA COMPLETA,NaN,...,SECRETARIA MUNICIPAL DE EDUCACAO,SME,DIRETORIA REGIONAL DE EDUCACAO PIRITUBA/JARAGUA,LAPA,LAPA,NaN,FEMININO,1942,BRANCA,NAO


In [10]:
cargos = df['REF_CARGO_BAS'].unique()

In [11]:
for cargo in cargos:
    if "appgg" in str(cargo).lower():
        print(cargo)

APPGG2
APPGG1
APPGG5
APPGG6
APPGG4
APPGG3


In [12]:
appggs = df[df['REF_CARGO_BAS'].str.contains('APPGG')].reset_index(drop=True)

In [13]:
appggs.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,6394604,3,CLAUDIO AGUIAR ALMEIDA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SMC,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SE,SE,NaN,MASCULINO,1963,PRETA,NAO
1,7575491,2,MAURICIO DA SILVA CORREIA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG1,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,SECRETARIA MUNICIPAL DE GESTAO,SE,SE,PREFEITURA DO MUNICIPIO DE ITAJAI,MASCULINO,1986,BRANCA,NAO
2,7718543,6,MARCIA MIYUKI ISHIKAWA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE HABITACAO,SEHAB,DEPARTAMENTO DE PLANEJAMENTO HABITACIONAL,SE,SE,NaN,FEMININO,1978,BRANCA,NAO
3,7794720,2,TIAGO ROSA MACHADO,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,ASSESSOR III,...,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,SEME,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,MOEMA,VILA MARIANA,NaN,MASCULINO,1983,BRANCA,NAO
4,7840501,2,THAIS ROBERTO DA SILVA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG5,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SMDHC,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SE,SE,NaN,FEMININO,1977,PRETA,NAO


In [14]:
appggs = appggs[['REGISTRO', 'NOME', 'REF_CARGO_BAS', 'SIGLA', 'DATA_INICIO_EXERC']]

renomear_cols = {
    'REGISTRO' : 'rf',
    'NOME' : 'nome',
    'REF_CARGO_BAS' : 'cargo_base',
    'SIGLA' : 'secretaria_dez_2025',
    'DATA_INICIO_EXERC' : 'dt_inicio_exercicio'
}

appggs.rename(renomear_cols, axis=1, inplace=True)


In [15]:
appggs['nivel_carreira'] = appggs['cargo_base'].str.extract(r'(\d+)$').astype(int)

In [16]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,01/10/2021,2
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,03/11/2021,1
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,08/12/2021,2
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,05/01/2022,2
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,29/11/2017,5


In [17]:
appggs.shape

(170, 6)

In [18]:
recem_nomeados = [{
            'rf' : str(i+1).zfill(7),
            'nome' : f'recem_nomeado_{i+1}',
            'cargo_base' : 'APPGG1',
            'secretaria_dez_2025' : 'NA',
            'nivel_carreira' : 1,
            'dt_inicio_exercicio' : '01/03/2026'
            }
            for i in range(30)
            ]

recem_nomeados = pd.DataFrame(recem_nomeados)

In [19]:
recem_nomeados.head()

,rf,nome,cargo_base,secretaria_dez_2025,nivel_carreira,dt_inicio_exercicio
0,0000001,recem_nomeado_1,APPGG1,NA,1,01/03/2026
1,0000002,recem_nomeado_2,APPGG1,NA,1,01/03/2026
2,0000003,recem_nomeado_3,APPGG1,NA,1,01/03/2026
3,0000004,recem_nomeado_4,APPGG1,NA,1,01/03/2026
4,0000005,recem_nomeado_5,APPGG1,NA,1,01/03/2026


In [20]:
recem_nomeados.shape

(30, 6)

In [21]:
appggs = pd.concat([appggs, recem_nomeados])

In [22]:
appggs.shape

(200, 6)

In [23]:
#vou colocar como datetime porque precisamos calcular com base na data
appggs['dt_inicio_exercicio'] = pd.to_datetime(appggs['dt_inicio_exercicio'], format ="%d/%m/%Y")


#inclusive ja vou definir se a pessoa contribui para o regime proprio (IPREM) ou nao
appggs['contribui_rpps'] = appggs['dt_inicio_exercicio'].dt.year<2018

In [24]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


# Situaçao atual

Nessa seção calculamos a situação atual de dispêndio com a carreira de APPGGs por parte da Prefeitura, considerando apenas os vencimentos e os encargos que estão relacionados aos vencimentos, como contribuição previdenciaria.

In [25]:
tabela_atual = {
    1 : 13208.14,
    2 : 14528.96,
    3 : 14892.18,
    4 : 15264.48,
    5 : 15646.09,
    6 : 16037.24,
    7 : 17961.73,
    8 : 18410.77,
    9 : 18871.04,
    10 : 19342.80,
    11 : 19826.39,
    12 : 21869.74,
    13 : 22416.47,
    14 : 22976.89,
    15 : 23551.31
}

In [26]:
appggs_atual = appggs.copy(deep=True)

In [27]:
appggs_atual['vencimento'] = appggs_atual['nivel_carreira'].map(tabela_atual)

In [28]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09


### Terço adicional de férias e décimo terceiro

Nesse caso vamos dividir ambos por 12 são gastos anuais e nossa base é mensal.

In [29]:
def calcular_decimo_terceiro(row):

    return round(row['vencimento']/12, 2)

def calcular_terco_ferias(row):

    return round((row['vencimento']/3)/12, 2)

In [30]:
appggs_atual['decimo_terceiro'] = appggs_atual.apply(calcular_decimo_terceiro, axis=1)

appggs_atual['terco_ferias'] = appggs_atual.apply(calcular_terco_ferias, axis=1)

In [31]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61


### Vale alimentação

O vale alimentação só é devido para quem ganha menos de 10 salários minimos. Como com o reajuste essa proporcao pode diminuir, ele deve entrar na conta apesar de ser um valor fixo.

In [32]:
def calcular_va(row):
    SALARIO_MINIMO = 1621
    VALOR_VALE_ALIMENTACAO = 325.25

    if row['vencimento'] <= SALARIO_MINIMO*10:
        return VALOR_VALE_ALIMENTACAO
    return 0

In [33]:
appggs_atual['vale_alimentacao'] = appggs_atual.apply(calcular_va, axis=1)

In [34]:
appggs_atual.sample(3)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao
29,8358923,JOAO PAULO DE BRITO GRECO,APPGG6,SEGES,2016-06-23,6,True,16037.24,1336.44,445.48,325.25
24,0000025,recem_nomeado_25,APPGG1,NA,2026-03-01,1,False,13208.14,1100.68,366.89,325.25
60,8359504,LEONARDO BARBOSA OLIVEIRA,APPGG6,SEPLAN,2016-07-11,6,True,16037.24,1336.44,445.48,325.25


### Contribuicao previdenciaria

Para calcular a contribuicao previdenciaria precisamos saber se a pessoa entrou antes de 2018 (ou seja, se é do IPREM) ou depois (se é do SAMPAPREV)

In [35]:
def valor_iprem(row):

    CONTRIBUICAO_IPREM = 0.28

    if not row['contribui_rpps']:
        return 0
    
    # o iprem nao considera o terço adicional de férias
    valor_base = row['vencimento'] + row['decimo_terceiro']
    
    return round(valor_base * CONTRIBUICAO_IPREM, 2)

In [36]:
appggs_atual['contribuicao_iprem'] = appggs_atual.apply(valor_iprem, axis=1)

In [37]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem
42,8359105,GABRIELA PINHEIRO LIMA CHABBOUH,APPGG6,SVMA,2016-07-04,6,True,16037.24,1336.44,445.48,325.25,4864.63
62,8375992,NATALIA AUGUSTO,APPGG2,SEGES,2022-01-07,2,False,14528.96,1210.75,403.58,325.25,0.00
45,8359164,HENRIQUE POUGY,APPGG6,SEPLAN,2016-07-01,6,True,16037.24,1336.44,445.48,325.25,4864.63
140,9411721,LUIS PEDRO POLESI DE CASTRO,APPGG1,SF,2024-07-15,1,False,13208.14,1100.68,366.89,325.25,0.00


In [38]:
def contribuicao_inss(row):

    ALIQUOTA_INSS = 0.21
    TETO_INSS = 8475.55

    if row['contribui_rpps']:
        return 0
    
    # o inss vai sobre todos os vencimentos, incluso terco adicional
    valor_base = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']

    #só para validar o teto - no caso de APPGG da na mesma porque tá todo mundo acima, mas pra deixar a funcao correta
    if valor_base > TETO_INSS:
        valor_base = TETO_INSS
        
    return round(valor_base*ALIQUOTA_INSS, 2)
    


In [39]:
appggs_atual['contribuicao_inss'] = appggs_atual.apply(contribuicao_inss, axis=1)

In [40]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61,325.25,4745.98,0.00
43,8359148,MARIANA SUCUPIRA GOMES,APPGG6,SEHAB,2016-06-30,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00
102,8897034,GABRIEL DE SOUZA TROVO,APPGG2,SMDET,2021-09-27,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87
155,9411925,TIAGO MILAGRES MIRANDA,APPGG1,SVMA,2024-08-05,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87


In [41]:
def previdencia_complementar(row):

    TETO_INSS = 8475.55
    ALIQUOTA_COMPLEMENTAR = 0.075
    if row['contribui_rpps']:
        return 0
    
    valor_base = row['vencimento'] + row['decimo_terceiro']
    #só contribui o que é acima do teto
    valor_base = valor_base - TETO_INSS

    return round(valor_base * ALIQUOTA_COMPLEMENTAR, 2)



In [42]:
appggs_atual['previdencia_complementar'] = appggs_atual.apply(previdencia_complementar, axis=1)

In [43]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar
167,9445994,FELIPE LARA VOGEL,APPGG1,SMT,2024-10-21,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.50
50,8359393,NATHALIA LEONE MARCO,APPGG6,PGM,2016-07-05,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00,0.00
93,8894230,CASSIANA MONTESIAO DE SOUSA,APPGG2,SEGES,2021-09-23,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81
30,8358931,LIA PALM,APPGG6,SMDET,2016-06-24,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00,0.00


## Valor total

Agora calculamos o valor total e salvamos os dados

In [44]:
def valor_total_prefeitura(row):

    vencimentos = row['vencimento'] + row['decimo_terceiro'] + row['terco_ferias']
    auxilios = row['vale_alimentacao']
    previdencia = row['contribuicao_iprem'] + row['contribuicao_inss'] + row['previdencia_complementar']

    return round(vencimentos+auxilios+previdencia, 2)

In [45]:
appggs_atual['valor_total_prefeitura'] = appggs_atual.apply(valor_total_prefeitura, axis=1)

In [46]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.50,17218.33
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61,325.25,4745.98,0.00,0.00,22455.77


In [47]:
appggs_atual.to_csv('situacao_atual.csv', sep=';')

# Proposta equiparação AMCI

Agora calculamos quais seriam as contribuicoes da Prefeitura com base na equiparação com a tabela dos AMCI

In [48]:
tabela_amci = {
    1 : 16840.38,
    2 : 17682.41,
    3 : 18036.04,
    4 : 18396.78,
    5 : 18764.71,
    6 : 19139.99,
    7 : 20097.00,
    8 : 20498.94,
    9 : 20908.92,
    10 : 21327.10,
    11 : 21753.64,
    12 : 22841.33,
    13 : 23183.95,
    14 : 23531.72,
    15 : 23884.67
}

In [49]:
proposta_amci = appggs.copy(deep=True)

In [50]:
proposta_amci.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


Agora vamos calcular rapidamente o que fizemos na seção anterior, qual seja:
1. o novo vencimento,
2. o décimo terceiro,
3. o terço adicional de férias,
4. o auxilio alimentacao,
5. as contribuicoes previdenciarias

In [51]:
#vencimento
proposta_amci['vencimento'] = proposta_amci['nivel_carreira'].map(tabela_amci)
#ferias e decimo terceiro
proposta_amci['decimo_terceiro'] = proposta_amci.apply(calcular_decimo_terceiro, axis=1)
proposta_amci['terco_ferias'] = proposta_amci.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_amci['vale_alimentacao'] = proposta_amci.apply(calcular_va, axis=1)
#previdencia
proposta_amci['contribuicao_iprem'] = proposta_amci.apply(valor_iprem, axis=1)
proposta_amci['contribuicao_inss'] = proposta_amci.apply(contribuicao_inss, axis=1)
proposta_amci['previdencia_complementar'] = proposta_amci.apply(previdencia_complementar, axis=1)

#VALOR TOTAL PROPOSTA
proposta_amci['valor_total_prefeitura'] = proposta_amci.apply(valor_total_prefeitura, axis=1)

In [52]:
proposta_amci.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,16840.38,1403.37,467.79,0,0.00,1779.87,732.62,21224.03
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,18764.71,1563.73,521.24,0,5691.96,0.00,0.00,26541.64


In [53]:
proposta_amci.to_csv('proposta_amci.csv', sep=';')

# Proposta reajuste inflação
### Reajuste tabela original

In [54]:
tabela_original = {
    1 : 9000.00,
    2 : 10080.00,
    3 : 10684.80,
    4 : 11325.89,
    5 : 12005.44,
    6 : 12725.77,
    7 : 13998.34,
    8 : 14698.26,
    9 : 15433.17,
    10 : 16204.83,
    11 : 17015.08,
    12 : 18716.58,
    13 : 19558.83,
    14 : 20438.98,
    15 : 21358.73
}

In [55]:
calculadora_inflacao = CalculadoraFatorInflacao('ipca')

In [56]:
# nesse momento o ultimo valor do IPCA disponível é para janeiro de 2026
fator_ipca = calculadora_inflacao('01/06/2016', '01/01/2026')

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F06%2F2016&dataFinal=01%2F01%2F2026


In [57]:
fator_ipca

1.5887292779061

In [58]:
tabela_atualizada = {nivel : round(valor*fator_ipca, 2) for nivel, valor in tabela_original.items()}
tabela_atualizada

{1: 14298.56,
 2: 16014.39,
 3: 16975.25,
 4: 17993.77,
 5: 19073.39,
 6: 20217.8,
 7: 22239.57,
 8: 23351.56,
 9: 24519.13,
 10: 25745.09,
 11: 27032.36,
 12: 29735.58,
 13: 31073.69,
 14: 32472.01,
 15: 33933.24}

In [59]:
proposta_ipca = appggs.copy(deep=True)

In [60]:
#vencimento
proposta_ipca['vencimento'] = proposta_ipca['nivel_carreira'].map(tabela_atualizada)
#ferias e decimo terceiro
proposta_ipca['decimo_terceiro'] = proposta_ipca.apply(calcular_decimo_terceiro, axis=1)
proposta_ipca['terco_ferias'] = proposta_ipca.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_ipca['vale_alimentacao'] = proposta_ipca.apply(calcular_va, axis=1)
#previdencia
proposta_ipca['contribuicao_iprem'] = proposta_ipca.apply(valor_iprem, axis=1)
proposta_ipca['contribuicao_inss'] = proposta_ipca.apply(contribuicao_inss, axis=1)
proposta_ipca['previdencia_complementar'] = proposta_ipca.apply(previdencia_complementar, axis=1)

#VALOR TOTAL PROPOSTA
proposta_ipca['valor_total_prefeitura'] = proposta_ipca.apply(valor_total_prefeitura, axis=1)

In [61]:
proposta_ipca.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,16014.39,1334.53,444.84,325.25,0.0,1779.87,665.50,20564.38
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,14298.56,1191.55,397.18,325.25,0.0,1779.87,526.09,18518.50
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,16014.39,1334.53,444.84,325.25,0.0,1779.87,665.50,20564.38
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,16014.39,1334.53,444.84,325.25,0.0,1779.87,665.50,20564.38
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,19073.39,1589.45,529.82,0.00,5785.6,0.00,0.00,26978.26


In [62]:
proposta_ipca.to_csv('proposta_ipca.csv', sep=';')

### Reajuste tabela Gestão Nunes

In [63]:
# agora consideramos junho de 2021 como o ano base
# vamos usar a tabela original como base porque só teve reajuste de 0.001% nesse período então nem compensa atualizar

In [64]:
fator_ipca_nunes = calculadora_inflacao('01/06/2021', '01/01/2026')

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F06%2F2021&dataFinal=01%2F01%2F2026


In [65]:
fator_ipca_nunes

1.294126076227806

In [66]:
tabela_atualizada_nunes = {nivel : round(valor*fator_ipca_nunes, 2) for nivel, valor in tabela_original.items()}
tabela_atualizada_nunes

{1: 11647.13,
 2: 13044.79,
 3: 13827.48,
 4: 14657.13,
 5: 15536.55,
 6: 16468.75,
 7: 18115.62,
 8: 19021.4,
 9: 19972.47,
 10: 20971.09,
 11: 22019.66,
 12: 24221.61,
 13: 25311.59,
 14: 26450.62,
 15: 27640.89}

Só a comparação com a tabela atual permite identificar que alguns níveis já tiveram reajuste até acima da inflação da gestão. Então no caso vamos manter esses niveis como estão.

In [67]:
tabela_atual

{1: 13208.14,
 2: 14528.96,
 3: 14892.18,
 4: 15264.48,
 5: 15646.09,
 6: 16037.24,
 7: 17961.73,
 8: 18410.77,
 9: 18871.04,
 10: 19342.8,
 11: 19826.39,
 12: 21869.74,
 13: 22416.47,
 14: 22976.89,
 15: 23551.31}

In [68]:
tabela_atualizada_nunes = {nivel : valor if valor>tabela_atual[nivel] else tabela_atual[nivel]
                           for nivel, valor in tabela_atualizada_nunes.items()}

In [69]:
tabela_atualizada_nunes

{1: 13208.14,
 2: 14528.96,
 3: 14892.18,
 4: 15264.48,
 5: 15646.09,
 6: 16468.75,
 7: 18115.62,
 8: 19021.4,
 9: 19972.47,
 10: 20971.09,
 11: 22019.66,
 12: 24221.61,
 13: 25311.59,
 14: 26450.62,
 15: 27640.89}

In [70]:
proposta_ipca_nunes = appggs.copy(deep=True)

In [71]:
#vencimento
proposta_ipca_nunes['vencimento'] = proposta_ipca_nunes['nivel_carreira'].map(tabela_atualizada_nunes)
#ferias e decimo terceiro
proposta_ipca_nunes['decimo_terceiro'] = proposta_ipca_nunes.apply(calcular_decimo_terceiro, axis=1)
proposta_ipca_nunes['terco_ferias'] = proposta_ipca_nunes.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_ipca_nunes['vale_alimentacao'] = proposta_ipca_nunes.apply(calcular_va, axis=1)
#previdencia
proposta_ipca_nunes['contribuicao_iprem'] = proposta_ipca_nunes.apply(valor_iprem, axis=1)
proposta_ipca_nunes['contribuicao_inss'] = proposta_ipca_nunes.apply(contribuicao_inss, axis=1)
proposta_ipca_nunes['previdencia_complementar'] = proposta_ipca_nunes.apply(previdencia_complementar, axis=1)

#VALOR TOTAL PROPOSTA
proposta_ipca_nunes['valor_total_prefeitura'] = proposta_ipca_nunes.apply(valor_total_prefeitura, axis=1)

In [72]:
proposta_ipca_nunes.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.50,17218.33
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61,325.25,4745.98,0.00,0.00,22455.77


In [73]:
proposta_ipca_nunes.to_csv('proposta_ipca_nunes.csv', sep=';')

In [74]:
def tabela_to_pandas(tabela:dict, tabela_name:str):

    dados = []
    for nivel, val in tabela.items():
        dados.append([tabela_name, nivel, val])

    return pd.DataFrame(dados, columns = ['nome_tabela', 'nivel', 'vencimento'])

In [75]:
tabelas = {
    'atual' : tabela_atual,
    'original' : tabela_original,
    'amci' : tabela_amci,
    'original_atualizada_ipca' : tabela_atualizada,
    'original_atualizada_ipca_nunes' : tabela_atualizada_nunes

}

In [76]:
dfs_tabelas = [tabela_to_pandas(tabela, tabela_name) for tabela_name, tabela in tabelas.items()]
df_tabelas = pd.concat(dfs_tabelas)

df_tabelas

,nome_tabela,nivel,vencimento
0,atual,1,13208.14
1,atual,2,14528.96
2,atual,3,14892.18
3,atual,4,15264.48
4,atual,5,15646.09
...,...,...,...
10,original_atualizada_ipca_nunes,11,22019.66
11,original_atualizada_ipca_nunes,12,24221.61
12,original_atualizada_ipca_nunes,13,25311.59
13,original_atualizada_ipca_nunes,14,26450.62


In [77]:
df_tabelas.to_csv('tabelas_vencimentos_utilizadas.csv', sep=';')

# Agregações

Agora fazemos as agregações para analisar o impacto orçamentário

In [78]:
def total_mensal_por_nivel(servidores, nome_proposta:str):

    df = servidores.groupby('nivel_carreira')['valor_total_prefeitura'].sum().reset_index()
    df['id_proposta'] = nome_proposta
    return df

In [79]:
total_atual = total_mensal_por_nivel(appggs_atual, 'situacao_atual')
total_atual

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1448128.82,situacao_atual
1,2,1146386.42,situacao_atual
2,3,21389.42,situacao_atual
3,4,109580.05,situacao_atual
4,5,334506.02,situacao_atual
5,6,782307.36,situacao_atual


In [80]:
total_amci = total_mensal_por_nivel(proposta_amci, 'unificacao_amci')
total_amci

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1785414.28,unificacao_amci
1,2,1355909.22,unificacao_amci
2,3,25510.97,unificacao_amci
3,4,130106.10,unificacao_amci
4,5,395101.48,unificacao_amci
5,6,920463.64,unificacao_amci


In [81]:
total_ipca = total_mensal_por_nivel(proposta_ipca, 'reajuste_ipca')
total_ipca

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1557585.27,reajuste_ipca
1,2,1254427.18,reajuste_ipca
2,3,24010.54,reajuste_ipca
3,4,127255.95,reajuste_ipca
4,5,401582.22,reajuste_ipca
5,6,972296.64,reajuste_ipca


In [82]:
total_ipca_nunes = total_mensal_por_nivel(proposta_ipca_nunes, 'reajuste_ipca_gestao_nunes')
total_ipca_nunes

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1448128.82,reajuste_ipca_gestao_nunes
1,2,1146386.42,reajuste_ipca_gestao_nunes
2,3,21389.42,reajuste_ipca_gestao_nunes
3,4,109580.05,reajuste_ipca_gestao_nunes
4,5,334506.02,reajuste_ipca_gestao_nunes
5,6,792000.76,reajuste_ipca_gestao_nunes


In [83]:
def impacto_mensal(situacao_atual, proposta):

    join = pd.merge(situacao_atual, proposta, on='nivel_carreira', suffixes=('_atual', '_proposta'))
    join['impacto_mensal'] = join['valor_total_prefeitura_proposta'] - join['valor_total_prefeitura_atual']

    return join



In [84]:
impacto_mensal_amci = impacto_mensal(total_atual, total_amci)
impacto_mensal_ipca = impacto_mensal(total_atual, total_ipca)
impacto_mensal_ipca_nunes = impacto_mensal(total_atual, total_ipca_nunes)

In [85]:
impacto_mensal_amci.to_csv('impacto_mensal_amci.csv', sep=';')
impacto_mensal_amci

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1448128.82,situacao_atual,1785414.28,unificacao_amci,337285.46
1,2,1146386.42,situacao_atual,1355909.22,unificacao_amci,209522.80
2,3,21389.42,situacao_atual,25510.97,unificacao_amci,4121.55
3,4,109580.05,situacao_atual,130106.10,unificacao_amci,20526.05
4,5,334506.02,situacao_atual,395101.48,unificacao_amci,60595.46
5,6,782307.36,situacao_atual,920463.64,unificacao_amci,138156.28


In [86]:
impacto_mensal_ipca.to_csv('impacto_mensal_ipca.csv', sep=';')
impacto_mensal_ipca

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1448128.82,situacao_atual,1557585.27,reajuste_ipca,109456.45
1,2,1146386.42,situacao_atual,1254427.18,reajuste_ipca,108040.76
2,3,21389.42,situacao_atual,24010.54,reajuste_ipca,2621.12
3,4,109580.05,situacao_atual,127255.95,reajuste_ipca,17675.90
4,5,334506.02,situacao_atual,401582.22,reajuste_ipca,67076.20
5,6,782307.36,situacao_atual,972296.64,reajuste_ipca,189989.28


In [87]:
impacto_mensal_ipca_nunes.to_csv('impacto_mensal_ipca_nunes.csv', sep=';')
impacto_mensal_ipca_nunes

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1448128.82,situacao_atual,1448128.82,reajuste_ipca_gestao_nunes,0.0
1,2,1146386.42,situacao_atual,1146386.42,reajuste_ipca_gestao_nunes,0.0
2,3,21389.42,situacao_atual,21389.42,reajuste_ipca_gestao_nunes,0.0
3,4,109580.05,situacao_atual,109580.05,reajuste_ipca_gestao_nunes,0.0
4,5,334506.02,situacao_atual,334506.02,reajuste_ipca_gestao_nunes,0.0
5,6,782307.36,situacao_atual,792000.76,reajuste_ipca_gestao_nunes,9693.4


In [88]:
impactos_anuais = {
'reajuste_ipca_gestao_nunes' : round(impacto_mensal_ipca_nunes['impacto_mensal'].sum()*12, 2),
'reajuste_ipca' : round(impacto_mensal_ipca['impacto_mensal'].sum()*12, 2),
'reajuste_amci' : round(impacto_mensal_amci['impacto_mensal'].sum()*12, 2),
}

In [89]:
tabela_to_pandas(impactos_anuais, 'impactos_anuais').drop('nome_tabela', axis=1).to_csv('impactos_anuais.csv', sep=';')